<br>
<font>
<div dir=ltr align=center>
<div dir=ltr align=center>
<font color=0F5298 size=7>
    Artificial Intelligence <br>
<font color=2565AE size=5>
    Computer Engineering Department <br>
    Arash Marioriyad<br>
    Spring 2026<br>
<font color=3C99D size=5>
    Practical HomeWork 2<br>
    Nonogram<br>
<font color=696880 size=4>
    Foad Kheirabady

`Full Name:`محمد مهدی مرادی

`Student ID:`403106681

# Introduction

In this notebook, you will implement several algorithms for solving **Constraint Satisfaction Problems (CSPs)**.
We begin with a **baseline brute-force solver** that assigns values blindly and uses **backtracking** when a constraint is violated.
You will then progressively enhance it with:

- **Forward Checking** – pruning inconsistent domain values after each assignment,
- **Minimum Remaining Values (MRV)** – always choosing the most constrained variable next,
- **Arc Consistency (AC-3)** – enforcing global consistency across all variable domains.

## What is a CSP?

A **Constraint Satisfaction Problem** is defined by three components:

- **Variables:** the unknowns we need to assign values to.
- **Domains:** the set of possible values each variable can take.
- **Constraints:** rules restricting which combinations of values are valid.

## Nonogram as a CSP

**Nonograms** (also known as Picross or Griddlers) are grid-based logic puzzles.
Each cell in an $N \times M$ grid must be filled ($1$) or left empty ($0$).
The valid pattern is determined by **clues** given for each row and column.

A clue is a sequence of integers indicating the lengths of consecutive filled-cell blocks, in left-to-right
(or top-to-bottom) order, with **at least one empty cell** between consecutive blocks.

**Example:** the clue `[2, 1]` for a row of length 5 means: a block of 2 filled cells, at least one gap, then a block of 1 filled cell.
Valid arrangements include `11010`, `11001`, `01101`.

We model this as a CSP:

- **Variables:** every cell $(i, j)$ in the $N \times M$ grid.
- **Domain:** $\{0, 1\}$ for each cell ($0$ = empty, $1$ = filled).
- **Constraints:** for each row $i$, the sequence of filled cells must match `row_clues[i]`;
  for each column $j$, it must match `col_clues[j]`.
- **Neighbors** of $(i, j)$: all other cells sharing the same row or column.

An example Nonogram puzzle is shown below.

![Example Nonogram](Nonogram-of-a-Dolphin.webp)

## Helper Functions in `utils.py`

Two utility functions are provided and are **free to use** throughout this notebook:

| Function | Description |
|---|---|
| `get_valid_arrangements(clue, length)` | Returns **all** valid binary lists of the given length satisfying the clue. |
| `is_line_possible(values, clue)` | Given a partial line (list with $-1$ for unassigned), returns `True` if the clue can still be satisfied. |

Study these helpers before starting — they are the key building blocks for every solver.


## Grading
| Chapter | Points |
|---|---|
| Baseline Solver | 10 |
| Forward Checking | 30 |
| MRV | 10 |
| AC-3 | 50 |

**NOTE 1:** All value assignments **must** use the **`state.assign`** function.
Direct modification of the grid outside this function may result in loss of the chapter's grade.

**NOTE 2:** Runtime optimization is encouraged but not required — the key metric is the **number of assignments**.

**NOTE 3:** You earn chapter points only if your solver makes **fewer assignments** than the previous solver.

**NOTE 4:** Two examples are used throughout this notebook to evaluate each solver:
- **First Example (15×15)** — a moderately complex puzzle used for **testing and debugging**.
  It is intentionally designed to take 5–10 seconds on the baseline solver, giving a clear
  signal that each successive solver is making meaningful improvements in assignments and backtracks.
- **Second Example (15×20)** — the **main benchmark puzzle**. This is the primary puzzle
  on which solvers are graded. It is significantly harder and may take a long time on weaker solvers.
  
> ⚠️ **Important:** Each solver must show a measurable reduction in the number of assignments
> on **both** examples compared to the previous solver. Pay close attention to the second example
> — it is the one that determines your grade.

In [2]:
import time
from collections import deque

import numpy as np

from models import NonogramCSP, Cell, StepLogger
from utils import (
    build_nonogram_csp, make_animation,
    get_valid_arrangements, is_line_possible,
    ROW_CLUES_1, COL_CLUES_1,
    ROW_CLUES_2, COL_CLUES_2
)


# Baseline Solver — 10 Points

In this chapter, implement the **baseline backtracking solver**.

Strategy:
- Iterate through cells in row-major order (left to right, top to bottom).
- For each unassigned cell, try each value in $\{0, 1\}$.
- If the value violates any constraint, skip it.
- If neither value works, **backtrack** to the previous assignment.

Do **not** use forward checking, MRV, or any domain pruning at this stage.

**Consistency check:** after tentatively assigning a value, call `is_line_possible`
on the current row and the current column. If either fails, the value is invalid.

## The `ValuesState` Class

`ValuesState` tracks the current assignment.  The grid is a NumPy array initialised
to $-1$ (unassigned). Always use **`state.assign`** to update a cell — it handles
the step logger so the animation remains consistent.

In [3]:
class ValuesState:
    assignment: np.ndarray   # shape (N, M), dtype int,  -1 = unassigned

    def __init__(self, csp: NonogramCSP):
        self.assignment = np.full((csp.N, csp.M), -1, dtype=int)

    def assign(self, cell: Cell, v: int, csp: NonogramCSP, logger: StepLogger):
        """
        Assign value v to cell.
        Pass v = -1 to undo an assignment (backtrack).
        """
        if v == -1:
            logger.log("backtrack", cell, int(self.assignment[cell]))
        else:
            logger.log("assign", cell, v)
        self.assignment[cell] = v

In [4]:
def solve_baseline(csp: NonogramCSP, state: ValuesState, logger: StepLogger) -> bool:

    def is_consistent(cell: Cell, value: int) -> bool:
        row = cell[0]
        col = cell[1]
        
        state.assignment[row, col] = value
        
        current_row = state.assignment[row, :]
        current_col = state.assignment[:, col]
        
        row_is_ok = is_line_possible(current_row, csp.row_clues[row])
        col_is_ok = is_line_possible(current_col, csp.col_clues[col])
        
        state.assignment[row, col] = -1
        
        if row_is_ok == True and col_is_ok == True:
            return True
        else:
            return False

    def backtrack() -> bool:
        unassigned_cell = None
        for cell in csp.variables:
            if state.assignment[cell] == -1:
                unassigned_cell = cell
                break
                
        if unassigned_cell == None:
            return True
            
        for value in [0, 1]:
            if is_consistent(unassigned_cell, value) == True:
                state.assign(unassigned_cell, value, csp, logger)
                
                result = backtrack()
                if result == True:
                    return True
                    
                state.assign(unassigned_cell, -1, csp, logger)
                
        return False

    return backtrack()

In [5]:
# Set to True to also run the second (harder) 15×20 example.
# Warning: it may take significantly longer to solve.
RUN_SECOND_EXAMPLE = True

def run_example(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = ValuesState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_baseline(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    # Return csp + logger so the animation section can use them
    return csp, logger


csp1, logger1 = run_example("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2, logger2 = None, None
if RUN_SECOND_EXAMPLE:
    csp2, logger2 = run_example("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)



First Example (15×15)
Solved:      True
Runtime:     1.368 seconds
Assignments: 780
Prunes:      0
Backtracks:  555

Second Example (15×20)
Solved:      True
Runtime:     113.631 seconds
Assignments: 19317
Prunes:      0
Backtracks:  19017


In [ ]:
make_animation(csp1, logger1, fps=5, last_k=150, stride=1)

if csp2 is not None:
    make_animation(csp2, logger2, fps=5, last_k=150, stride=1)

# Forward Checking — 30 Points

In this chapter, extend the baseline solver with **forward checking**.

After each assignment to cell $(r, c)$, inspect **row $r$** and **column $c$**:

1. Compute all valid arrangements of that line consistent with the current assignment
   (use `get_valid_arrangements`, then filter by the current partial state).
2. For each **unassigned** cell in the line, determine which values still appear
   in at least one valid arrangement.
3. **Prune** values that never appear.
4. If any unassigned cell's domain becomes **empty**, the current branch fails — backtrack immediately.

Do **not** use the MRV heuristic yet; keep assigning cells in row-major order.

## The `DomainValuesState` Class

`DomainValuesState` extends `ValuesState` with per-cell `domains`.  
Always use **`state.prune`** and **`state.unprune`** to change domains — never
modify `domains` directly. These methods also update the step logger.

In [6]:
class DomainValuesState(ValuesState):
    domains: dict   # Cell -> set of remaining values  {0, 1}

    def __init__(self, csp: NonogramCSP):
        super().__init__(csp)
        self.domains = {cell: {0, 1} for cell in csp.variables}

    def prune(self, cell: Cell, v: int, logger: StepLogger):
        self.domains[cell].discard(v)
        logger.log("prune", cell, v)

    def unprune(self, cell: Cell, v: int):
        self.domains[cell].add(v)

In [ ]:
def solve_forward_check(csp: NonogramCSP, state: DomainValuesState, logger: StepLogger) -> bool:
    def check_line(cells: list, clue: tuple, length: int):
        cur_assign = [int(state.assignment[cell]) for cell in cells]
        valid_assign = get_valid_arrangements(clue, length)
        filter_assign = []
        for assign in valid_assign:
            flag = True
            for i in range(length):
                if (cur_assign[i] != -1 and cur_assign[i] != assign[i]) or (assign[i] not in state.domains[cells[i]]):
                    flag = False
                    break
            if flag:
                filter_assign.append(assign)

        if len(filter_assign) == 0:
            return None
        
        prune_list = []
        for i, cell in enumerate(cells):
            if cur_assign[i] != -1:
                continue
            
            cur_dom = state.domains[cell]
            exi_dom = set()
            for assign in filter_assign:
                exi_dom.add(assign[i])

            if len(exi_dom) == 0:
                return None
            for dom in cur_dom:
                if dom not in exi_dom:
                    prune_list.append((cell, dom))

        return prune_list

    def forward_check(r: int, c: int):
        r_len = csp.M
        c_len = csp.N
        r_check = check_line([(r, col) for col in range(r_len)], csp.row_clues[r], r_len)
        if r_check == None:
            return None
        c_check = check_line([(row, c) for row in range(c_len)], csp.col_clues[c], c_len)
        if c_check == None:
            return None
        pruned_list = r_check + c_check
        return pruned_list

    def backtrack() -> bool:
        if len(csp.variables) == 0:
            return True

        sel_cell = csp.variables[0]
        for value in sorted(state.domains[sel_cell]):
            state.assign(sel_cell, value, csp, logger)
            csp.variables.remove(sel_cell)

            r, c = sel_cell
            fw_check = forward_check(r, c)
            if fw_check != None:
                for cell, value in fw_check:
                    state.prune(cell, value, logger)

                if backtrack():
                    return True
                
                for cell, value in fw_check:
                    state.unprune(cell, value)

            csp.variables.insert(0, sel_cell)
            state.assign(sel_cell, -1, csp, logger)

        return False

    return backtrack()

In [8]:
def run_example_fc(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = DomainValuesState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_forward_check(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_fc, logger1_fc = run_example_fc("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_fc, logger2_fc = None, None
if RUN_SECOND_EXAMPLE:
    csp2_fc, logger2_fc = run_example_fc("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)




First Example (15×15)
Solved:      True
Runtime:     0.329 seconds
Assignments: 306
Prunes:      212
Backtracks:  81

Second Example (15×20)
Solved:      True
Runtime:     22.376 seconds
Assignments: 7735
Prunes:      8887
Backtracks:  7435


In [ ]:
make_animation(csp1_fc, logger1_fc, fps=5, last_k=150, stride=1)

if csp2_fc is not None:
    make_animation(csp2_fc, logger2_fc, fps=5, last_k=150, stride=1)

Discuss your results. In what ways did forward checking improve over the baseline? Is it always faster?

`Your Answer:`در الگوریتم forward checking به دلیل هرس کردن یه سری دامنه از گره های مختلف باعث میشه مسیر های اشتباه سریعتر شناسایی شوند در صورتی که در الگوریتم پایه مسیر های اشتباه ممکن است مقدار زیادی جستجو انجام شود بعد شناسایی شوند که این زمان الگوریتم را بسیار بالا میبرد.

و اینکه همواره سریعتر نیست،چون چک کردن دامنه ها به خودی خود زمان اضافه میگیرد و در یک سری کیس ممکن است این چک کردن زمان الگوریتم را نسبت به حالت پایه بیشتر کند.

# Minimum Remaining Values (MRV) — 10 Points

In this chapter, add the **MRV heuristic** on top of forward checking.

Instead of assigning cells in fixed row-major order, always pick the **unassigned cell
with the fewest remaining domain values**.  Targeting the most constrained cell first
tends to expose failures earlier, pruning large branches of the search tree.

Continue using **one-step forward checking** (exactly as in Chapter 2) after each assignment.

## The `MRVState` Class

`MRVState` extends `DomainValuesState` with a `num_remaining_values` array for fast
MRV lookups.  Pruning and unpruning automatically keep this count consistent.

In [11]:
class MRVState(DomainValuesState):
    num_remaining_values: np.ndarray   # shape (N, M)

    def __init__(self, csp: NonogramCSP):
        super().__init__(csp)
        self.num_remaining_values = np.full((csp.N, csp.M), 2, dtype=int)

    def prune(self, cell: Cell, v: int, logger: StepLogger):
        self.domains[cell].discard(v)
        self.num_remaining_values[cell] -= 1
        logger.log("prune", cell, v)

    def unprune(self, cell: Cell, v: int):
        self.domains[cell].add(v)
        self.num_remaining_values[cell] += 1

In [12]:
def solve_mrv(csp: NonogramCSP, state: MRVState, logger: StepLogger) -> bool:

    def select_unassigned() -> Cell:
        best_cell = None
        min_values = 9999
        
        for cell in csp.variables:
            if state.assignment[cell] == -1:
                rem_vals = state.num_remaining_values[cell]
                if rem_vals < min_values:
                    min_values = rem_vals
                    best_cell = cell
                    
        return best_cell

    def check_line(cells: list, clue: tuple, length: int):
        current_values = []
        for cell in cells:
            val = int(state.assignment[cell])
            current_values.append(val)
            
        all_arrangements = get_valid_arrangements(clue, length)
        valid_arrangements = []
        
        for arr in all_arrangements:
            is_match = True
            for i in range(length):
                if current_values[i] != -1:
                    if current_values[i] != arr[i]:
                        is_match = False
                        break
            if is_match == True:
                valid_arrangements.append(arr)
                
        if len(valid_arrangements) == 0:
            return None
            
        pruned_list = []
        for i in range(length):
            cell = cells[i]
            if current_values[i] == -1:
                possible_values = set()
                for arr in valid_arrangements:
                    possible_values.add(arr[i])
                    
                domain_copy = list(state.domains[cell])
                for v in domain_copy:
                    if v not in possible_values:
                        state.prune(cell, v, logger)
                        pruned_list.append((cell, v))
                        
                if len(state.domains[cell]) == 0:
                    for item in pruned_list:
                        undo_cell = item[0]
                        undo_val = item[1]
                        state.unprune(undo_cell, undo_val)
                    return None
                    
        return pruned_list

    def forward_check(r: int, c: int):
        row_cells = []
        for j in range(csp.M):
            row_cells.append((r, j))
            
        row_pruned = check_line(row_cells, csp.row_clues[r], csp.M)
        
        if row_pruned == None:
            return None
            
        col_cells = []
        for i in range(csp.N):
            col_cells.append((i, c))
            
        col_pruned = check_line(col_cells, csp.col_clues[c], csp.N)
        
        if col_pruned == None:
            for item in row_pruned:
                cell = item[0]
                val = item[1]
                state.unprune(cell, val)
            return None
            
        all_pruned = []
        for item in row_pruned:
            all_pruned.append(item)
        for item in col_pruned:
            all_pruned.append(item)
            
        return all_pruned

    def backtrack() -> bool:
        unassigned_cell = select_unassigned()
        
        if unassigned_cell == None:
            return True
            
        for value in [0, 1]:
            if value in state.domains[unassigned_cell]:
                state.assign(unassigned_cell, value, csp, logger)
                
                pruned_items = forward_check(unassigned_cell[0], unassigned_cell[1])
                
                if pruned_items != None:
                    result = backtrack()
                    if result == True:
                        return True
                        
                state.assign(unassigned_cell, -1, csp, logger)
                
                if pruned_items != None:
                    for item in pruned_items:
                        cell = item[0]
                        val = item[1]
                        state.unprune(cell, val)
                        
        return False

    return backtrack()

In [13]:
def run_example_mrv(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = MRVState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_mrv(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_mrv, logger1_mrv = run_example_mrv("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_mrv, logger2_mrv = None, None
if RUN_SECOND_EXAMPLE:
    csp2_mrv, logger2_mrv = run_example_mrv("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)



First Example (15×15)
Solved:      True
Runtime:     0.225 seconds
Assignments: 231
Prunes:      174
Backtracks:  6

Second Example (15×20)
Solved:      True
Runtime:     0.922 seconds
Assignments: 300
Prunes:      299
Backtracks:  0


In [ ]:
make_animation(csp1_mrv, logger1_mrv, fps=5, last_k=150, stride=1)

if csp2_mrv is not None:
    make_animation(csp2_mrv, logger2_mrv, fps=5, last_k=150, stride=1)

Compare your results to the previous solvers. How did MRV affect assignments and backtracks? Why does choosing the most constrained cell first help?

`Your Answer:`الگوریتم forward checking صرفا فضای جستجو را کوچکتر میکند و ترکیب آن با MRV میتواند ما را به سمت مسیر های بهینه تر با سرعت بیشتری ببرد در نتیجه سرعت الگوریتم به صورت چشم گیری بهتر میشود(مخصوصا در جدول دوم که از حدود یک دقیقه به حدود یک ثانیه رسید)

دلیل اینکه انتخاب کردن most constrained cell به ما کمک میکند این است که مسیر هایی که با شکست مواجه میشوند را زودتر شناسایی میکنیم و اینکه ممکن است الگوریتم های قبلی به سرعت یک درخت جستجوی بسیار بزرگی تشکیل بدهند اگر ابتدا سراغ خانه ها با انتخاب های بیشتر برویم.

# Arc Consistency (AC-3) — 50 Points

In this chapter, integrate **AC-3** into the MRV solver.

### Why AC-3?

Forward checking only re-examines the **two lines** directly touched by the most recent
assignment.  AC-3 goes further: whenever a pruning changes a cell's domain, it re-queues
**all other lines containing that cell**, propagating the constraint like a cascade.
This often eliminates many values (or forces assignments) long before backtracking would
reach those cells.

### AC-3 for Nonogram

An **arc** here corresponds to a **line** — a complete row or column.  The AC-3 queue holds
`('row', i)` and `('col', j)` items.

**Algorithm:**

1. Seed the queue (all lines at startup; just the affected row and column after an assignment).
2. Dequeue a line and compute all valid arrangements consistent with current assignments and domains.
3. For each unassigned cell in the line, prune values absent from every valid arrangement.
4. If a cell's domain **changed**, add its **other-dimension** line back to the queue
   (`'col'` for a cell in a row, and `'row'` for a cell in a column).
5. If any domain becomes empty, return failure.

Continue using **MRV** for cell selection.

**Implementation note:** `ac3` should return `(success: bool, pruned: list)`.
On backtracking, restore all items in `pruned` with `state.unprune`.

**Hint:** call `ac3()` once before the first assignment to reduce domains globally.

In [14]:
def solve_ac3_mrv(csp: NonogramCSP, state: MRVState, logger: StepLogger) -> bool:

    def ac3(initial_queue=None):
        all_pruned = []
        queue = []
        
        if initial_queue == None:
            for i in range(csp.N):
                queue.append(('row', i))
            for j in range(csp.M):
                queue.append(('col', j))
        else:
            for item in initial_queue:
                queue.append(item)

        while len(queue) > 0:
            current_item = queue.pop(0)
            line_type = current_item[0]
            idx = current_item[1]
            
            cells = []
            clue = None
            length = 0
            
            if line_type == 'row':
                for j in range(csp.M):
                    cells.append((idx, j))
                clue = csp.row_clues[idx]
                length = csp.M
            else:
                for i in range(csp.N):
                    cells.append((i, idx))
                clue = csp.col_clues[idx]
                length = csp.N
                
            current_values = []
            for cell in cells:
                val = int(state.assignment[cell])
                current_values.append(val)
                
            all_arrangements = get_valid_arrangements(clue, length)
            valid_arrangements = []
            
            for arr in all_arrangements:
                is_match = True
                for i in range(length):
                    cell = cells[i]
                    if current_values[i] != -1:
                        if current_values[i] != arr[i]:
                            is_match = False
                            break
                    else:
                        if arr[i] not in state.domains[cell]:
                            is_match = False
                            break
                if is_match == True:
                    valid_arrangements.append(arr)
                    
            if len(valid_arrangements) == 0:
                return (False, all_pruned)
                
            for i in range(length):
                cell = cells[i]
                if current_values[i] == -1:
                    possible_values = set()
                    for arr in valid_arrangements:
                        possible_values.add(arr[i])
                        
                    domain_copy = list(state.domains[cell])
                    domain_changed = False
                    
                    for v in domain_copy:
                        if v not in possible_values:
                            state.prune(cell, v, logger)
                            all_pruned.append((cell, v))
                            domain_changed = True
                            
                    if len(state.domains[cell]) == 0:
                        return (False, all_pruned)
                        
                    if domain_changed == True:
                        if line_type == 'row':
                            new_item = ('col', cell[1])
                            if new_item not in queue:
                                queue.append(new_item)
                        else:
                            new_item = ('row', cell[0])
                            if new_item not in queue:
                                queue.append(new_item)

        return (True, all_pruned)

    def select_unassigned() -> Cell:
        best_cell = None
        min_values = 9999
        
        for cell in csp.variables:
            if state.assignment[cell] == -1:
                rem_vals = state.num_remaining_values[cell]
                if rem_vals < min_values:
                    min_values = rem_vals
                    best_cell = cell
                    
        return best_cell

    def backtrack() -> bool:
        unassigned_cell = select_unassigned()
        
        if unassigned_cell == None:
            return True
            
        for value in [0, 1]:
            if value in state.domains[unassigned_cell]:
                state.assign(unassigned_cell, value, csp, logger)
                
                initial_q = []
                initial_q.append(('row', unassigned_cell[0]))
                initial_q.append(('col', unassigned_cell[1]))
                
                success, pruned_items = ac3(initial_q)
                
                if success == True:
                    result = backtrack()
                    if result == True:
                        return True
                        
                state.assign(unassigned_cell, -1, csp, logger)
                
                for item in pruned_items:
                    cell = item[0]
                    val = item[1]
                    state.unprune(cell, val)
                    
        return False

    success, _ = ac3()
    if success == False:
        return False

    return backtrack()

In [18]:
def run_example_ac3(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = MRVState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_ac3_mrv(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_ac3, logger1_ac3 = run_example_ac3("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_ac3, logger2_ac3 = None, None
if RUN_SECOND_EXAMPLE:
    csp2_ac3, logger2_ac3 = run_example_ac3("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)



First Example (15×15)
Solved:      True
Runtime:     0.424 seconds
Assignments: 226
Prunes:      172
Backtracks:  1

Second Example (15×20)
Solved:      True
Runtime:     1.109 seconds
Assignments: 300
Prunes:      300
Backtracks:  0


In [ ]:
make_animation(csp1_ac3, logger1_ac3, fps=5, last_k=150, stride=1)

if csp2_ac3 is not None:
    make_animation(csp2_ac3, logger2_ac3, fps=5, last_k=150, stride=1)

Discuss your results. Did AC-3 improve runtime compared to MRV alone? Is AC-3 always beneficial? What are its tradeoffs?

`Your Answer:`با اجرای AC-3 ممکن است که تعداد assignment ها کمتر شود ولی اجرا کردن خود این الگوریتم زمان زیادی میگیرد و لزوما بهتر نیست.چون که باید یک preprocess انجام بدهیم داریم زمان بیشتری میدیم تا تعداد assignment ها کمتر شود و این یک tradeoff بین زمان و تعداد assignment ها هستش

در جدول دوم چون 15 در 20 است حداقل 300 تا assignment داریم پس برای همین تعدادش کمتر نشده از الگوریتمMRV به AC-3